## Medical Reasoning LLM - Data Cleaning

### Objective

Prepare the medical reasoning dataset for downstream SFT and RLVR.

### Dataset

- Source: https://huggingface.co/datasets/FreedomIntelligence/medical-o1-reasoning-SFT
- Key fields:
    - Question
    - Complex_CoT
    - Response

### Pipeline

- Fetch Raw Dataset
- Train / Test Split
- Explore Training Set
- Duplicate Analysis
- Text Normalization

## Setup

In [ ]:
# Clone repository
!git clone https://github.com/xueqingnie/ml-ai-portfolio.git

Cloning into 'ml-ai-portfolio'...
remote: Enumerating objects: 77, done.
remote: Counting objects: 100% (77/77), done.
remote: Compressing objects: 100% (66/66), done.
remote: Total 77 (delta 21), reused 5 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (77/77), 83.03 KiB | 3.08 MiB/s, done.
Resolving deltas: 100% (21/21), done.


In [1]:
%cd /content/ml-ai-portfolio/01-medical-reasoning-llm

/content/ml-ai-portfolio/01-medical-reasoning-llm


In [2]:
# Sync with GitHub
!git pull origin main

From https://github.com/xueqingnie/ml-ai-portfolio
 * branch            main       -> FETCH_HEAD
Already up to date.


In [3]:
!git status

On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


## Fetch Raw Dataset

In [4]:
from src.data import fetch_hf_dataset

#import inspect
#print(inspect.signature(fetch_hf_dataset))

In [5]:
dataset = fetch_hf_dataset(
    "FreedomIntelligence/medical-o1-reasoning-SFT",
    config="en",
    split="train"
)
df = dataset.to_pandas()

In [7]:
print("Dataset shape:", df.shape)
print(df.columns.tolist())

Dataset shape: (19704, 3)
['Question', 'Complex_CoT', 'Response']


In [6]:
df.head()

,Question,Complex_CoT,Response
0,Given the symptoms of sudden weakness in the l...,"Okay, let's see what's going on here. We've go...",The specific cardiac abnormality most likely t...
1,A 33-year-old woman is brought to the emergenc...,"Okay, let's figure out what's going on here. A...","In this scenario, the most likely anatomical s..."
2,A 61-year-old woman with a long history of inv...,"Okay, let's think about this step by step. The...",Cystometry in this case of stress urinary inco...
3,A 45-year-old man with a history of alcohol us...,"Alright, let’s break this down. We have a 45-y...",Considering the clinical presentation of sudde...
4,A 45-year-old man presents with symptoms inclu...,"Okay, so here's a 45-year-old guy who's experi...",Based on the clinical findings presented—wide-...


## Train / Test Split

In [8]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(
    n_splits = 1,
    test_size = 0.1,
    random_state = 42
)

train_index, test_index = next(
    gss.split(df, groups=df["Question"])
)

train = df.iloc[train_index].copy()
test = df.iloc[test_index].copy()

In [12]:
print("Training:", len(train), "Training ratio:", len(train) / len(df))
print("Test:", len(test), "Test ratio:", len(test) / len(df))

Training: 17735 Training ratio: 0.9000710515631344
Test: 1969 Test ratio: 0.09992894843686562


In [15]:
# Check for question leakage
question_overlap = set(train["Question"]) & set(test["Question"])
print(len(question_overlap))

0


## Training Set Exploration

In [16]:
train.describe()

,Question,Complex_CoT,Response
count,17735,17735,17735
unique,17711,17735,17734
top,What is the name of the classification propose...,"Alright, let's think this through. The symptom...",D. The first statement is false and the second...
freq,3,1,2


In [17]:
train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 17735 entries, 0 to 19703
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Question     17735 non-null  object
 1   Complex_CoT  17735 non-null  object
 2   Response     17735 non-null  object
dtypes: object(3)
memory usage: 554.2+ KB


In [23]:
# Check for missing values

print(train.isnull().sum())

Question       0
Complex_CoT    0
Response       0
dtype: int64


In [24]:
# Check for empty strings and whitespace-only strings

text_columns = ["Question", "Complex_CoT", "Response"]

for col in text_columns:
    empty_count = train[col].str.strip().eq("").sum()
    print(f"{col}: {empty_count} empty and whitespace-only strings")

Question: 0 empty and whitespace-only strings
Complex_CoT: 0 empty and whitespace-only strings
Response: 0 empty and whitespace-only strings


## Duplicate Analysis

In [26]:
# Check for duplicated rows and fields

print("Duplicated rows:", train.duplicated().sum())

for col in text_columns:
    duplicated_count = train[col].duplicated().sum()
    print(f"{col}: {duplicated_count} duplicated values")

Duplicated rows: 0
Question: 24 duplicated values
Complex_CoT: 0 duplicated values
Response: 1 duplicated values


In [28]:
duplicated_questions = train[train["Question"].duplicated(keep=False)]

duplicated_questions.sort_values("Question")

,Question,Complex_CoT,Response
4527,"After a laparoscopic cholecystectomy, at which...","Okay, so we're talking about where a biliary s...","After a laparoscopic cholecystectomy, a biliar..."
14702,"After a laparoscopic cholecystectomy, at which...","Alright, so we're looking at what happens afte...","After a laparoscopic cholecystectomy, a biliar..."
6273,At what age does a child typically begin to si...,So when do kids start to sit with a little hel...,"Typically, a child begins to sit with support,..."
12740,At what age does a child typically begin to si...,Let's think about when a child starts doing ce...,"Typically, a child begins to sit with support,..."
14949,At what age does a child typically begin to si...,"Okay, let's think about when babies typically ...",Babies typically begin to sit with support aro...
2491,At what left atrial pressure does pulmonary ed...,"Alright, so we're trying to figure out when pu...",Pulmonary edema generally begins to appear whe...
3724,At what left atrial pressure does pulmonary ed...,"So, let's think about pulmonary edema for a mo...",Pulmonary edema typically begins to appear whe...
5361,In a 3-week-old child presenting with an abdom...,Let's think about what could cause an abdomina...,In a 3-week-old child presenting with an abdom...
13144,In a 3-week-old child presenting with an abdom...,A 3-week-old baby with an abdominal mass—what ...,In a 3-week-old child presenting with an abdom...
5620,To which layers of the lateral geniculate nucl...,"Okay, so let's think about the visual pathway ...",Fibers from the contralateral nasal hemiretina...


### Deduplication Conclusion

No fully duplicated rows were identified in the training set. Although 24 questions appeared more than once, these were retained because duplicate questions may correspond to distinct reasoning and response pairs. Therefore, no deduplication was performed at this stage.

In [29]:
# Text length analysis

train[text_columns].apply(
    lambda x: x.str.len()
).describe()

,Question,Complex_CoT,Response
count,17735.000000,17735.000000,17735.000000
mean,298.429546,1909.101551,639.553707
std,212.640074,409.621171,327.893964
min,51.000000,937.000000,4.000000
25%,163.000000,1622.000000,435.000000
50%,244.000000,1842.000000,570.000000
75%,360.000000,2132.000000,753.000000
max,2380.000000,5535.000000,3999.000000


In [33]:
train.loc[train["Question"].str.len().nsmallest(5).index]

,Question,Complex_CoT,Response
3215,In which conditions is sympathectomy not indic...,"Alright, let's think this through. Sympathecto...",Sympathectomy is not indicated for several con...
3871,Which bacterial species exhibit Dienes' phenom...,"Okay, let's think this through. I've heard abo...",Dienes' phenomenon is primarily associated wit...
5187,What functions or reflexes are lost in brain d...,"Alright, let's see what happens with brain dea...","In brain death, all functions and reflexes gov..."
14924,What are two key characteristics of laryngomal...,"Okay, let's think through this. What are we tr...",Two key characteristics of laryngomalacia are ...
17565,Which tests are used to assess vestibular func...,"Alright, so when it comes to understanding how...","To assess vestibular function, several special..."


In [32]:
train.loc[train["Complex_CoT"].str.len().nsmallest(5).index]

,Question,Complex_CoT,Response
19318,Calculate the relative risk for a population i...,"Alright, let's start by figuring out what this...",The relative risk for the population is 5. The...
19279,A 32-year-old male has a clean wound without l...,"Okay, let's think about tetanus prophylaxis fo...",For a 32-year-old male with a clean wound who ...
6389,"In a population of 100,000 under surveillance ...",I'm trying to figure out the Annual Parasite I...,To calculate the Annual Parasite Incidence (AP...
14307,A 59-year-old male presents with dimness of ne...,"Okay, let's think about this—this guy is 59 an...",The appropriate next step in managing this 59-...
15750,At what age is sexual development considered p...,"Alright, let's think about when kids usually s...","Sexual development, specifically breast and pu..."


In [34]:
train.loc[train["Response"].str.len().nsmallest(5).index]

,Question,Complex_CoT,Response
3329,Consider the following in a new born : Hea rat...,"Alright, let's figure out the Apgar score for ...",B. 3
16699,Consider the following statements in respect o...,Let's take a look at these statements one at a...,D. 3
19242,Father has a blood group B : Mother has AB : C...,"Okay, let's figure this out. We know blood gro...",A. 0
2497,Induction of labor by amniotomy can lead to th...,"Okay, so I'm thinking about the risks involved...",A. ad
2618,"True about endotracheal cuff – a) Low-volume, ...","Okay, let's look into the details of endotrach...",B. ac


## Text Normalization

In [39]:
# Check for leading/trailing whitespace

for col in text_columns:
    leading = train[col].str.startswith(" ").sum()
    trailing = train[col].str.endswith(" ").sum()

    print(f"{col}:")
    print(f"  Leading spaces: {leading}")
    print(f"  Trailing spaces: {trailing}")

Question:
  Leading spaces: 0
  Trailing spaces: 0
Complex_CoT:
  Leading spaces: 0
  Trailing spaces: 0
Response:
  Leading spaces: 0
  Trailing spaces: 0


In [37]:
# Strip leading/trailing whitespace

for col in text_columns:
    train[col] = train[col].str.strip()

In [38]:
# Re-check for empty strings after stripping leading/trailing whitespace

for col in text_columns:
    empty_count = train[col].str.strip().eq("").sum()
    print(f"{col}: {empty_count} empty strings")

Question: 0 empty strings
Complex_CoT: 0 empty strings
Response: 0 empty strings
